# 27. 후보 C에 22b 피처 RF·ET 섞기 (H27)

## 배경
- 후보 C(22b XGB 0.7 + LR 0.3)가 LB 0.3650, XGB 시드 5개 평균을 더해 0.3660 (11/12 현재 최고)
- 지금까지 LB로 이어진 것: 모델 교체(09), 튜닝(10), 섞기(14), 치환 비율(22b, 24b)
- 14에서는 XGB + LR + RF 같은 비율이 OOF 최고였음 (예전 피처). **22b 피처로 만든 RF·ET는 아직 섞어 본 적 없음**
- C + 24b LR은 기각 (같은 치환 정보가 겹침) → 이번에는 피처는 그대로 두고 **모델 종류**를 늘림

## 가설
- H27: 22b 피처(any + ratio + 치환 비율 상위 100)로 학습한 RF·ET를 C에 섞으면 C보다 오른다

## 검증 방법
- C는 22 노트북과 같은 구성: 22b XGB(10 설정) 0.7 + LR(any, C=0.1, balanced) 0.3
- RF·ET: 14와 같은 설정(500그루, sqrt, min_samples_leaf 2, balanced_subsample), 피처만 22b
  - 치환 상위 100개 선택은 22와 같은 `TopNSubs`로 Pipeline 안에서 fold 학습 부분으로만 결정
- 섞기: (1 − w) × C + w × 추가 모델
  - 고정 비율 w = 0.1, 0.2, 0.3 결과를 모두 표시
  - **교차 적합**: fold k의 w는 나머지 4개 fold로만 고르고 fold k에서 채점 (16과 같은 방식)
- 채택 기준: C보다 5 fold 중 4개 이상 상승
- 진단(채택 판단에는 안 씀): C와 예측이 다른 환자 비율(OOF vs test), test COAD·STES 예측 비율
- test는 예측에만 사용. 제출 파일은 설아님 Windows 데스크탑에서만 생성 (`SAVE`)
- **기준 2개로 비교**: ① C (XGB 시드 42 하나) ② C + XGB 시드 5개 평균 (42, 0, 1, 2, 3 → 17·재현 스크립트와 같은 시드, 현재 LB 최고 0.3660의 구성)
  - 제출 후보는 ②(시드 평균) 위에 RF·ET를 섞은 것


In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, run_cv, macro_f1, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
FOLDS = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

Linux | CPU x86_64 | xgboost 3.4.1


#### 22b 피처 (22 노트북과 같은 계산)

In [2]:
def sub_ratio_matrix(df):
    # 환자별 미스센스 치환 종류 비율 (한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    # fold 학습 부분에서 비율 합이 큰 상위 n개 치환 열만 남김 (22와 동일)
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
X22b    = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1)
X22b_te = pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)
print('22b 피처:', X22b.shape, '| 치환 종류', S_tr.shape[1])

22b 피처: (6201, 4632) | 치환 종류 241


#### 모델: 22b XGB, LR(any), 22b RF, 22b ET
- 확률은 `submissions/cache27/`에 저장 (다시 실행하면 학습 없이 불러옴, git 제외)

In [3]:
CACHE = Path('../submissions/cache27'); CACHE.mkdir(parents=True, exist_ok=True)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
TREE_KW = dict(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample',
               random_state=SEED, n_jobs=-1)
MODELS = {
    'XGB_22b': (lambda: make_pipeline(TopNSubs(100), BalancedXGB(**BEST)), X22b, X22b_te),
    'LR_any':  (lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
                F_tr['any'].astype('float32'), F_te['any'].astype('float32')),
    'RF_22b':  (lambda: make_pipeline(TopNSubs(100), RandomForestClassifier(**TREE_KW)), X22b, X22b_te),
    'ET_22b':  (lambda: make_pipeline(TopNSubs(100), ExtraTreesClassifier(**TREE_KW)), X22b, X22b_te),
}
fold_scores = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in FOLDS])
OOF, TE = {}, {}
for name, (make, X, X_te) in MODELS.items():
    p = CACHE / f'{name}.npz'
    if p.exists():
        z = np.load(p); OOF[name], TE[name] = z['oof'], z['test']
    else:
        t = time.time()
        r = run_cv(make, X, y, X_te, folds=FOLDS, verbose=False)
        OOF[name], TE[name] = r['oof_proba'], r['test_proba']
        np.savez(p, oof=OOF[name], test=TE[name])
        print(f'  {name}: 학습 {time.time() - t:.0f}s')
    s = fold_scores(OOF[name]); print(f'{name}: CV {s.mean():.4f} ± {s.std():.4f}')

XGB_22b: CV 0.4489 ± 0.0068
LR_any: CV 0.3818 ± 0.0064
RF_22b: CV 0.4286 ± 0.0118
ET_22b: CV 0.4119 ± 0.0142


#### XGB 시드 5개 평균 (현재 LB 최고 구성)
- 시드 42는 위 `XGB_22b`, 나머지 4개(0, 1, 2, 3)는 같은 설정에서 `random_state`만 바꿈

In [4]:
SEEDS = [42, 0, 1, 2, 3]
xo, xt = [OOF['XGB_22b']], [TE['XGB_22b']]
for s in SEEDS[1:]:
    p = CACHE / f'XGB_22b_s{s}.npz'
    if p.exists():
        z = np.load(p); o, t_ = z['oof'], z['test']
    else:
        t = time.time()
        r = run_cv(lambda: make_pipeline(TopNSubs(100), BalancedXGB(**{**BEST, 'random_state': s})), X22b, y, X22b_te, folds=FOLDS, verbose=False)
        o, t_ = r['oof_proba'], r['test_proba']; np.savez(p, oof=o, test=t_)
        print(f'  seed {s}: 학습 {time.time() - t:.0f}s')
    print(f'seed {s}: CV {fold_scores(o).mean():.4f}')
    xo.append(o); xt.append(t_)
OOF['XGB_22b_5seed'], TE['XGB_22b_5seed'] = np.mean(xo, axis=0), np.mean(xt, axis=0)
s = fold_scores(OOF['XGB_22b_5seed']); print(f'XGB 22b 시드 5개 평균: CV {s.mean():.4f} ± {s.std():.4f}')

seed 0: CV 0.4526
seed 1: CV 0.4520
seed 2: CV 0.4477
seed 3: CV 0.4522
XGB 22b 시드 5개 평균: CV 0.4579 ± 0.0086


#### 비교: C vs C + RF / ET (고정 비율 + 교차 적합)

In [5]:
ADD = {'RF': (OOF['RF_22b'], TE['RF_22b']), 'ET': (OOF['ET_22b'], TE['ET_22b']),
       'RF + ET 평균': ((OOF['RF_22b'] + OOF['ET_22b']) / 2, (TE['RF_22b'] + TE['ET_22b']) / 2)}
GRID = [0, 0.1, 0.2, 0.3]
idx = np.arange(len(y))
f1 = lambda rows, P: macro_f1(y[rows], P[rows].argmax(1))
mix = lambda w, A, B: (1 - w) * A + w * B
BASES = {'C (시드 1개)':       (0.7 * OOF['XGB_22b'] + 0.3 * OOF['LR_any'], 0.7 * TE['XGB_22b'] + 0.3 * TE['LR_any']),
         'C + 시드 5개 평균': (0.7 * OOF['XGB_22b_5seed'] + 0.3 * OOF['LR_any'], 0.7 * TE['XGB_22b_5seed'] + 0.3 * TE['LR_any'])}

rows, CHOSEN = [], {}
for bname, (C_oof, C_te) in BASES.items():
    base = fold_scores(C_oof)
    print(f'{bname}: CV {base.mean():.4f} ± {base.std():.4f} | fold별 {base.round(4)}')
    for name, (a_oof, a_te) in ADD.items():
        for w in GRID[1:]:
            s = fold_scores(mix(w, C_oof, a_oof)); d = s - base
            rows.append({'기준': bname, '추가 모델': name, 'w': w, '방식': '고정', 'CV': s.mean(), '기준 대비': d.mean(), '기준보다 높은 fold': f'{(d > 0).sum()}/5'})
        sc, ws = [], []
        for _, va in FOLDS:                       # 교차 적합: fold k의 w는 나머지 fold로만 선택
            tr = np.setdiff1d(idx, va)
            w = max(GRID, key=lambda g: f1(tr, mix(g, C_oof, a_oof))); ws.append(w)
            sc.append(f1(va, mix(w, C_oof, a_oof)))
        d = np.array(sc) - base
        CHOSEN[(bname, name)] = (float(np.median(ws)), d, np.mean(sc))
        rows.append({'기준': bname, '추가 모델': name, 'w': str(ws), '방식': '교차 적합', 'CV': np.mean(sc), '기준 대비': d.mean(), '기준보다 높은 fold': f'{(d > 0).sum()}/5'})
res = pd.DataFrame(rows)
res.round(4)

C (시드 1개): CV 0.4613 ± 0.0094 | fold별 [0.4567 0.452  0.468  0.4535 0.4765]


C + 시드 5개 평균: CV 0.4634 ± 0.0106 | fold별 [0.4493 0.4563 0.4776 0.4607 0.4733]


,기준,추가 모델,w,방식,CV,기준 대비,기준보다 높은 fold
0,C (시드 1개),RF,0.1,고정,0.4629,0.0015,4/5
1,C (시드 1개),RF,0.2,고정,0.4653,0.0040,5/5
2,C (시드 1개),RF,0.3,고정,0.4651,0.0038,4/5
3,C (시드 1개),RF,"[0.2, 0.3, 0.3, 0.2, 0.3]",교차 적합,0.4645,0.0032,4/5
4,C (시드 1개),ET,0.1,고정,0.4646,0.0032,4/5
5,C (시드 1개),ET,0.2,고정,0.4648,0.0035,4/5
6,C (시드 1개),ET,0.3,고정,0.4643,0.0030,4/5
7,C (시드 1개),ET,"[0.2, 0.1, 0.1, 0.2, 0.1]",교차 적합,0.4641,0.0027,4/5
8,C (시드 1개),RF + ET 평균,0.1,고정,0.4644,0.0031,4/5
9,C (시드 1개),RF + ET 평균,0.2,고정,0.4644,0.0031,4/5


#### 진단 (채택 판단에는 사용하지 않음)

In [6]:
dist = lambda p: pd.Series(le.inverse_transform(p.argmax(1))).value_counts(normalize=True) * 100
B5 = 'C + 시드 5개 평균'
C_oof, C_te = BASES[B5]
diag, tab = [], {'train': train.SUBCLASS.value_counts(normalize=True) * 100, 'C (시드 1개)': dist(BASES['C (시드 1개)'][1]), B5: dist(C_te)}
for name, (a_oof, a_te) in ADD.items():
    w = CHOSEN[(B5, name)][0] or 0.2               # 교차 적합 w의 중앙값 (0이면 참고용으로 0.2)
    o, t = mix(w, C_oof, a_oof), mix(w, C_te, a_te)
    diag.append({'조합': f'{B5} + {name} (w={w})', '기준과 다른 환자 OOF %': (o.argmax(1) != C_oof.argmax(1)).mean() * 100,
                 '기준과 다른 환자 test %': (t.argmax(1) != C_te.argmax(1)).mean() * 100})
    tab[f'+ {name}'] = dist(t)
print(pd.DataFrame(diag).set_index('조합').round(1))
pd.DataFrame(tab).fillna(0).sort_values(B5, ascending=False).round(1).head(8)

                                   기준과 다른 환자 OOF %  기준과 다른 환자 test %
조합                                                                  
C + 시드 5개 평균 + RF (w=0.3)                      3.6               3.1
C + 시드 5개 평균 + ET (w=0.3)                      5.2               5.6
C + 시드 5개 평균 + RF + ET 평균 (w=0.3)              3.8               3.9


,train,C (시드 1개),C + 시드 5개 평균,+ RF,+ ET,+ RF + ET 평균
STES,6.1,21.4,20.7,20.7,19.2,20.0
COAD,3.6,13.8,14.5,15.0,16.4,15.7
KIPAN,8.3,5.7,5.7,6.0,5.9,6.0
BRCA,12.7,5.1,5.1,5.2,5.4,5.3
OV,4.1,4.9,4.8,4.9,4.9,4.9
KIRC,5.4,3.6,3.8,3.7,3.7,3.8
PCPG,2.4,3.6,3.5,3.5,3.3,3.3
PRAD,4.3,3.3,3.5,3.3,3.6,3.5


#### 제출 파일 (채택된 조합만, 설아님 Windows 데스크탑에서 `SAVE = True`)

In [7]:
SAVE = False   # 팀 규칙: 제출 파일은 Windows 데스크탑에서만 생성
# 제출 후보는 '시드 5개 평균' 기준에서 채택 기준(4/5)을 넘은 조합만
adopted = {n: (w, cv) for (b, n), (w, d, cv) in CHOSEN.items() if b == B5 and (d > 0).sum() >= 4 and w > 0}
print('채택 (시드 5개 평균 기준):', {n: w for n, (w, cv) in adopted.items()} or '없음')
TAG = {'RF': 'RF', 'ET': 'ET', 'RF + ET 평균': 'RFET'}
if SAVE:
    for name, (w, cv) in adopted.items():
        print('저장:', save_submission(submission, mix(w, C_te, ADD[name][1]), le, name=f'blend_ratio22sub100_seed5_LR_plus_{TAG[name]}_w{w}', cv_score=cv))

채택 (시드 5개 평균 기준): {'ET': 0.3}


## 결과 (Linux 실행, XGB 점수는 데스크탑과 ±0.004 차이 가능)

**모델 단독 CV**: 22b XGB 0.4489 (시드 5개 평균 0.4579) / 22b RF 0.4286 / 22b ET 0.4119 / LR(any) 0.3818

**기준 ②: C + XGB 시드 5개 평균 (CV 0.4634, 현재 LB 최고 0.3660의 구성)**

| 추가 모델 | w | CV | 기준 대비 | 기준보다 높은 fold |
| :-- | :-: | :-: | :-: | :-: |
| RF | 0.2 (고정) | 0.4644 | +0.0009 | 4/5 |
| RF | 교차 적합 | 0.4641 | +0.0007 | 3/5 |
| ET | 0.2 (고정) | 0.4679 | +0.0045 | 5/5 |
| **ET** | **0.3 (교차 적합, 5개 fold 모두 0.3 선택)** | **0.4688** | **+0.0054** | **5/5** |
| RF + ET 평균 | 교차 적합 | 0.4660 | +0.0026 | 2/5 |

**기준 ①: C (시드 1개, CV 0.4613)**: RF w=0.2 +0.0040 (5/5), ET 교차 적합 +0.0027 (4/5), RF 교차 적합 +0.0032 (4/5)

- 시드 평균 기준에서는 **ET만 채택 기준 통과** (+0.0054, 5/5). RF는 +0.001 수준으로 사실상 차이 없음
- 기준에 따라 RF와 ET의 순서가 바뀜 (시드 1개: RF ≥ ET / 시드 평균: ET > RF) → **이득이 작고 흔들림 범위에 가까움** (기준의 fold 간 std 0.0106)
- ET의 w가 탐색 범위 끝(0.3)에서 골라짐 → 더 큰 w는 시도하지 않음 (ET 단독 CV가 0.41로 낮아 비중을 더 키우는 것은 위험)

**진단 (채택 판단에는 안 씀)**
- 기준과 예측이 다른 환자: OOF 5.2% / test 5.6% → 비슷하게 움직이고, 변화 폭이 작음 (LB도 크게 움직이지 않을 것)
- test 예측: STES 20.7% → 19.2%, COAD 14.5% → 16.4% (22b의 STES 쏠림이 조금 줄고 COAD가 조금 늘어남)

## 결론
- **H27 부분 채택**: 시드 5개 평균 C에 22b ET를 0.3 섞으면 CV +0.0054, 5/5 fold → 채택 기준 통과
- 다만 이득이 std보다 작고 기준에 따라 RF·ET 순서가 바뀌므로 **확실한 개선이라고 보기는 어려움** → 남은 제출 기회가 있을 때 시도해 볼 후보
- 제출 후보: **(C + 시드 5개 평균) 0.7 + 22b ET 0.3** = XGB 5seed 0.49 + LR 0.21 + ET 0.30
- 제출 파일은 설아님 Windows 데스크탑에서 `SAVE = True`로 생성 (`blend_ratio22sub100_seed5_LR_plus_ET_w0.3`)
